# 🌿 XAI Computational Efficiency Benchmarking
### Evaluating Saliency, Grad-CAM, LIME & SHAP on Lightweight Edge Models

This notebook clones the project from GitHub, installs dependencies, and runs the full CRISP-DM benchmarking pipeline.

> **Tip:** Go to `Runtime → Change runtime type → T4 GPU` for much faster SHAP & LIME runs.

---
**What this notebook does:**
1. ⚙️ Sets up the environment (clone repo + install packages)
2. 💾 Uploads Kaggle credentials and downloads real PlantVillage Tomato images
3. 🏋️ Fine-tunes MobileNetV3 / ShuffleNetV2 on real PlantVillage data
4. 🔍 Generates visual explanation maps (Saliency, Grad-CAM, LIME, SHAP)
5. 📊 Benchmarks each XAI method over **N runs** and plots the distributions

## Step 1 — Clone the repository

In [ ]:
import os

REPO_URL = "https://github.com/bonsoirval/explainability_impact"
REPO_DIR = "explainability_impact"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Step 2 — Install dependencies

This installs PyTorch, Captum, LIME, SHAP and the rest of the project requirements.
First run takes ~2–3 minutes.

In [ ]:
!pip install -q -r requirements.txt
print("✅ All dependencies installed.")

## Step 2b — Upload Kaggle credentials (required)

This pipeline downloads the **real PlantVillage Tomato dataset** from Kaggle.
You must provide a valid `kaggle.json` API token before running the pipeline.

1. Go to [kaggle.com/settings](https://www.kaggle.com/settings) → **API** → **Create New Token** → download `kaggle.json`
2. Run the cell below and select the downloaded file when prompted.


In [ ]:
# Upload your kaggle.json to authenticate with the Kaggle API
import os

if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
    from google.colab import files
    print("Upload your kaggle.json file:")
    uploaded = files.upload()
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    os.rename("kaggle.json", os.path.expanduser("~/.kaggle/kaggle.json"))
    os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
    print("kaggle.json saved.")
else:
    print("kaggle.json already present.")

## Step 3 — Configuration

Adjust these variables to control the experiment:

| Variable | Meaning |
|---|---|
| `MODEL` | `mobilenet_v3` or `shufflenet_v2` |
| `RUNS` | How many times each XAI method is timed (100, 500, 1000 …) |
| `EPOCHS` | Training epochs (1 is fine for benchmarking, the task is to measure XAI cost) |
| `METHODS` | Subset of methods to run, or all four |

In [ ]:
MODEL        = "mobilenet_v3"          # "mobilenet_v3" | "shufflenet_v2"
RUNS         = 10                       # change to 100 / 500 / 1000 as needed
EPOCHS       = 1
METHODS      = ["Saliency", "Grad-CAM", "LIME", "SHAP"]

# Real PlantVillage Tomato dataset from Kaggle:
CROP         = "Tomato"                 # crop prefix to filter class folders
IMAGE_TYPE   = "color"                  # "color" | "segmented" | "grayscale"

RESULTS_DIR  = "phase_5_evaluation/results"

print(f"Model        : {MODEL}")
print(f"Runs         : {RUNS}")
print(f"Epochs       : {EPOCHS}")
print(f"Methods      : {METHODS}")
print(f"Crop filter  : {CROP}")
print(f"Image type   : {IMAGE_TYPE}")

## Step 4 — Run the full CRISP-DM pipeline

This single cell runs all six phases end-to-end.  
With `RUNS=10` on a T4 GPU it should finish in **under 10 minutes**.

In [ ]:
MODEL        = "mobilenet_v3"          # "mobilenet_v3" | "shufflenet_v2"
RUNS         = 10                       # change to 100 / 500 / 1000 as needed
EPOCHS       = 1
METHODS      = ["Saliency", "Grad-CAM", "LIME", "SHAP"]

# Real PlantVillage Tomato dataset from Kaggle:
CROP         = "Tomato"                 # crop prefix to filter class folders
IMAGE_TYPE   = "color"                  # "color" | "segmented" | "grayscale"

RESULTS_DIR  = "phase_5_evaluation/results"

print(f"Model        : {MODEL}")
print(f"Runs         : {RUNS}")
print(f"Epochs       : {EPOCHS}")
print(f"Methods      : {METHODS}")
print(f"Crop filter  : {CROP}")
print(f"Image type   : {IMAGE_TYPE}")

## Step 5 — View the explanation maps

In [ ]:
from IPython.display import Image, display
import os

explanations_path = os.path.join(RESULTS_DIR, f"{MODEL}_explanations.png")
print("Explanation maps (Saliency | Grad-CAM | LIME | SHAP):")
display(Image(filename=explanations_path))

## Step 6 — View the benchmark distribution plots

Box plots + jittered individual run points for **Latency** and **Peak RAM** across all methods.

In [ ]:
dist_path = os.path.join(RESULTS_DIR, f"{MODEL}_benchmark_distributions.png")
print(f"Distribution plot ({RUNS} runs per method):")
display(Image(filename=dist_path))

## Step 7 — Inspect the aggregated benchmark table

In [ ]:
import pandas as pd

report_path = os.path.join(RESULTS_DIR, f"{MODEL}_benchmark_report.csv")
df = pd.read_csv(report_path)
print(f"\nAggregated benchmark summary ({RUNS} runs):")
df.style.format({
    'Latency Mean (s)' : '{:.4f}',
    'Latency Std (s)'  : '{:.4f}',
    'Peak RAM Mean (MB)' : '{:.2f}',
    'Peak VRAM Mean (MB)': '{:.2f}',
}).background_gradient(subset=['Latency Mean (s)'], cmap='YlOrRd')

## Step 8 — Inspect every individual run

The raw run-by-run CSV lets you plot your own graphs or inspect variance.

In [ ]:
raw_path = os.path.join(RESULTS_DIR, f"{MODEL}_raw_runs.csv")
raw_df = pd.read_csv(raw_path)
print(f"Total individual runs recorded: {len(raw_df)}")
raw_df

## (Optional) Step 9 — Run ShuffleNetV2 for comparison

Uncomment and run this cell to also benchmark ShuffleNetV2.

In [ ]:
methods_arg = " ".join(METHODS)

!python run_pipeline.py \\
    --model      {MODEL}       \\
    --epochs     {EPOCHS}      \\
    --runs       {RUNS}        \\
    --methods    {methods_arg} \\
    --crop       {CROP}        \\
    --image_type {IMAGE_TYPE}

---
## Summary of output files

| File | Description |
|------|-------------|
| `phase_5_evaluation/results/{model}_explanations.png` | Visual explanation maps |
| `phase_5_evaluation/results/{model}_benchmark_distributions.png` | Box plot distributions over N runs |
| `phase_5_evaluation/results/{model}_benchmark_report.csv` | Mean ± std aggregated stats |
| `phase_5_evaluation/results/{model}_raw_runs.csv` | Every individual run data point |
| `phase_6_deployment/{model}.onnx` | Exported ONNX model for edge deployment |